# Mitra Regressor — End-to-End Regression with Your Own Data

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/mitra-regressor-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mitra-regressor-pipeline/blob/main/tutorials/mitra_regressor_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-autogluon%2Fmitra--regressor-ffcc4d?style=flat)](https://huggingface.co/autogluon/mitra-regressor)
[![Upstream](https://img.shields.io/badge/Upstream-autogluon%2Fautogluon-181717?style=flat&logo=github&logoColor=white)](https://github.com/autogluon/autogluon)
[![arXiv](https://img.shields.io/badge/arXiv-2510.21204-b31b1b.svg)](https://arxiv.org/abs/2510.21204)

You have a labelled table (a few thousand rows, a few dozen columns, a quantity to predict) and you want a credible regressor today, without a hyperparameter search and without trusting a model file you cannot verify. **Mitra** is a tabular foundation model: it was pre-trained on millions of synthetic tables and predicts a new table's values *in context*, by reading your labelled rows as its prompt. There is nothing to train for the baseline; optionally, a short fine-tune on a GPU can adapt it further.

This notebook takes you from the pinned Mitra checkpoint (from the DIMER Model Repository, or the identical upstream release) to a verified, reusable predictor ZIP, with every shortcut that quietly ruins tabular evaluations guarded against along the way.

**By the end of this notebook you will be able to:**
- **Acquire and verify** a model checkpoint by immutable revision and SHA-256, and lock the loader to that verified copy so nothing else can be substituted.
- **Prepare tabular data for honest evaluation**: keep leakage-aware splits when you have them, know when a random holdout is acceptable, and catch duplicate rows and non-numeric or constant targets before they distort a metric.
- **Evaluate Mitra in-context**, read MAE and RMSE against the trivial mean-predictor baseline, and decide when fine-tuning is worth a GPU.
- **Export a predictor bundle** that reloads bit-for-bit and carries its own provenance, so a colleague can score new rows without this notebook.

**No DIMER Workbench access is required.** Data is processed in Google Colab, not by DIMER. Do not upload confidential, sensitive, or restricted data unless that environment is permitted.

## Prerequisites

- **Runtime:** any Colab runtime works for the default path (pretrained, in-context evaluation runs on CPU). A GPU is needed only for `RUN_FINE_TUNING`. With the bundled sample the default run typically takes a few minutes, most of it the dependency install and the 300 MB checkpoint download; that figure comes from development runs on a local GPU workstation, not from a measured Colab session.
- **Knowledge:** pandas basics; what a train/validation/test split is for. You do not need to know AutoGluon; every call it makes is explained where it happens.
- **Data:** nothing, to start. The bundled FreshRetailNet sample ships with leakage-aware splits. For your own data you need a CSV with a numeric target column and at least 50 rows, or your own `train.csv` / `val.csv` / `test.csv`.

**How to use this notebook.** Cells with a form on the right (`# @param`) are the knobs; change one and re-run from that cell down. Run everything in order the first time. Each step says what the next cell does and what to look for in its output. Steps 5 and 6 are gated on Step 4 having completed *in this session*, on purpose.


## 1. Install and inspect the runtime

Mitra ships as an extra of AutoGluon (`autogluon.tabular[mitra]`), which pins a compatible PyTorch range and can therefore **replace Colab's preinstalled `torch`**. That is normal here, and the cell reports the actual PyTorch and CUDA versions *after* installation instead of assuming the accelerator stack was left alone. Two things to look for:

- If `torch` was already imported in this session and pip changed its version, the cell stops and asks you to *Runtime ▸ Restart session*. An imported module does not pick up a new version underneath it.
- Pip may report conflicts for unrelated preinstalled packages (`torchvision`, `diffusers`, `gradio`). This tutorial does not use them; those warnings are noise.

**What to look for:** `AutoGluon: 1.5.0` and a `CUDA available:` line. `False` is fine for the default path; fine-tuning (Step 4) will then be disabled.


In [ ]:
import importlib.metadata as importlib_metadata
import sys

PREINSTALL_TORCH_VERSION = importlib_metadata.version('torch')
TORCH_WAS_IMPORTED = 'torch' in sys.modules
print('PyTorch before install:', PREINSTALL_TORCH_VERSION)

%pip install -q "autogluon.tabular[mitra]==1.5.0" "lightgbm>=4.0,<4.8"

INSTALLED_TORCH_VERSION = importlib_metadata.version('torch')
AUTOGLUON_VERSION = importlib_metadata.version('autogluon.tabular')
if TORCH_WAS_IMPORTED and INSTALLED_TORCH_VERSION != PREINSTALL_TORCH_VERSION:
    raise RuntimeError('pip changed PyTorch after it had already been imported. Use Runtime → Restart session, then run the notebook top-to-bottom.')

import torch

TORCH_VERSION = torch.__version__
TORCH_CUDA_VERSION = torch.version.cuda
CUDA_AVAILABLE_AFTER_INSTALL = torch.cuda.is_available()
print('AutoGluon:', AUTOGLUON_VERSION)
print('PyTorch after install:', TORCH_VERSION)
print('PyTorch CUDA build:', TORCH_CUDA_VERSION)
print('CUDA available:', CUDA_AVAILABLE_AFTER_INSTALL)
if not CUDA_AVAILABLE_AFTER_INSTALL:
    print('⚠ Pretrained/in-context evaluation can still run on CPU, but fine-tuning will be disabled.')

## 2. Acquire, verify, and lock the checkpoint

A model file is code you are about to run and weights you are about to trust. This step makes both claims checkable:

- **DIMER ZIP** uploads the ZIP distributed through the DIMER Model Repository, which contains `model.safetensors`; the matching pinned `config.json` is fetched from upstream.
- **Pinned upstream** fetches both files from the exact AutoGluon revision `5f277aa8…` on the Hugging Face Hub. A revision is an immutable commit; a model *name* is a branch that can change.

Both files are **SHA-256 verified** against digests recorded in this notebook, then staged into an isolated Hugging Face cache and marked offline. Finally the notebook asks Hugging Face to resolve the model *exactly as AutoGluon will* and refuses to continue unless both resolved files come from that verified snapshot **and still match the expected digests**. That last check matters more than it looks: it catches out-of-order execution where `huggingface_hub` was imported before the offline environment was configured, which would silently let a later download win.

If the lock check fails, use *Runtime ▸ Restart session* and run from Step 1 downward. Network requests use a finite timeout so outages fail clearly instead of hanging.

**What to look for:** two `✓ … verified` lines with digest prefixes `d8e75c62af0b…` and `2bc1ed5047f7…`, then `✓ Hugging Face resolver locked …`.


In [ ]:
import hashlib, json, os, random, shutil, urllib.request, zipfile
from pathlib import Path

MODEL_ID = 'autogluon/mitra-regressor'
PINNED_REVISION = '5f277aa8f69042d39d6ac3612aed18bb9279bd95'
EXPECTED_WEIGHTS_SHA256 = 'd8e75c62af0bec2fd404b0ad20a442d951d43ca6d331315cfcc0509b54f2c642'
EXPECTED_CONFIG_SHA256 = '2bc1ed5047f7c25368245e8ad32540a5fa28940b1ec05d3f1f454a09ff5384c1'
NETWORK_TIMEOUT_SECONDS = 30

HF_HOME = Path('/content/mitra-hf')
MODEL_DIR = Path('/content/mitra-model')
HF_HOME.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def fetch_pinned(name, dest):
    url = f'https://huggingface.co/{MODEL_ID}/resolve/{PINNED_REVISION}/{name}?download=true'
    print('Retrieving pinned', name)
    with urllib.request.urlopen(url, timeout=NETWORK_TIMEOUT_SECONDS) as r, open(dest, 'wb') as f:
        shutil.copyfileobj(r, f)

def verify(path, expected, label):
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(f'{label} checksum mismatch.\nExpected: {expected}\nActual:   {actual}')
    print(f'✓ {label} verified: {actual[:12]}…')
    return actual

def weights_from_dimer(dest):
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Upload exactly one DIMER ZIP or model.safetensors.')
    name, payload = next(iter(uploaded.items()))
    p = MODEL_DIR / Path(name).name
    p.write_bytes(payload)
    if p.suffix.lower() == '.safetensors':
        if p.resolve() != dest.resolve():
            shutil.copy2(p, dest)
        return
    if p.suffix.lower() != '.zip':
        raise ValueError('Expected a DIMER ZIP or model.safetensors.')
    with zipfile.ZipFile(p) as z:
        matches = [i for i in z.infolist() if not i.is_dir() and Path(i.filename).name == 'model.safetensors']
        if len(matches) != 1:
            raise RuntimeError(f'Expected one model.safetensors in the DIMER ZIP; found {len(matches)}.')
        with z.open(matches[0]) as src, open(dest, 'wb') as dst:
            shutil.copyfileobj(src, dst)

def install_offline_snapshot(weights, config, weights_digest):
    snapshot = weights_digest[:40]
    repo = HF_HOME / 'hub' / ('models--' + MODEL_ID.replace('/', '--'))
    snap = repo / 'snapshots' / snapshot
    refs = repo / 'refs'
    snap.mkdir(parents=True, exist_ok=True)
    refs.mkdir(parents=True, exist_ok=True)
    shutil.copy2(weights, snap / 'model.safetensors')
    shutil.copy2(config, snap / 'config.json')
    (refs / 'main').write_text(snapshot)
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    return snap

def assert_resolver_locked(snapshot):
    from huggingface_hub import hf_hub_download
    expected = {
        'model.safetensors': ((snapshot / 'model.safetensors').resolve(), EXPECTED_WEIGHTS_SHA256),
        'config.json': ((snapshot / 'config.json').resolve(), EXPECTED_CONFIG_SHA256),
    }
    for filename, (expected_path, expected_digest) in expected.items():
        try:
            resolved = Path(hf_hub_download(repo_id=MODEL_ID, filename=filename)).resolve()
        except Exception as exc:
            raise RuntimeError('Verified snapshot is staged but Hugging Face cannot resolve it offline. Use Runtime → Restart session and run the notebook top-to-bottom.') from exc
        if resolved != expected_path:
            raise RuntimeError(f'Offline checkpoint lock is not in effect for {filename}.\nExpected: {expected_path}\nResolved: {resolved}\nUse Runtime → Restart session and run the notebook top-to-bottom.')
        resolved_digest = sha256_file(resolved)
        if resolved_digest != expected_digest:
            raise RuntimeError(f'Resolved {filename} digest changed after staging.\nExpected: {expected_digest}\nActual:   {resolved_digest}')
    print('✓ Hugging Face resolver locked to the verified offline snapshot and digests.')

MODEL_SOURCE = 'Pinned upstream'  # @param ['DIMER ZIP', 'Pinned upstream']

weights_path = MODEL_DIR / 'model.safetensors'
config_path = MODEL_DIR / 'config.json'

if MODEL_SOURCE == 'DIMER ZIP':
    weights_from_dimer(weights_path)
    fetch_pinned('config.json', config_path)
else:
    fetch_pinned('model.safetensors', weights_path)
    fetch_pinned('config.json', config_path)

weights_digest = verify(weights_path, EXPECTED_WEIGHTS_SHA256, 'model.safetensors')
verify(config_path, EXPECTED_CONFIG_SHA256, 'config.json')
SNAPSHOT_PATH = install_offline_snapshot(weights_path, config_path, weights_digest)
print('✓ Offline snapshot:', SNAPSHOT_PATH)
assert_resolver_locked(SNAPSHOT_PATH)


## 3. Choose a dataset

If you do not have a dataset, choose **Sample dataset (FreshRetailNet)**. The bundled ZIP contains `train.csv`, `val.csv`, and `test.csv`; the notebook preserves those partitions instead of randomly re-splitting them. It has 4,180 training rows, 1,600 validation rows, 1,600 test rows, 17 features, and a continuous target: the `target` column holds the sale amount seven days ahead (training mean 0.96, median 0.60, standard deviation 1.27, range 0–16, so it is skewed with a long right tail). It is derived from FreshRetailNet-50K, redistributed under **CC BY 4.0** for tutorial and smoke-test use, not benchmarking, and pinned to an immutable repository revision.

[Read the sample DATASET_CARD.md](https://github.com/kurtvalcorza/mitra-regressor-pipeline/blob/5625a9eeca94b8c72b9ad1ec78d07ecbaa720903/examples/sample-data/DATASET_CARD.md)

**Why the split is preserved.** The sample is a *purged chronological split with an embargo*: training rows come before validation and test rows in time, with a gap between them. A random re-split would let the model see the future of the very series it is asked to predict, and its metrics would be flattering and useless. That is the single most common way tabular evaluations go wrong.

For your own data:

- **Upload CSV** creates a seeded random holdout and therefore assumes rows are approximately IID. Do **not** use this mode for time-dependent, grouped, panel, lagged, or rolling-window data unless random splitting is scientifically appropriate.
- **Upload pre-split train/val/test** preserves partitions you prepared externally, which is the safer choice for temporal or grouped data or any workflow with an embargo/purge rule. Feature columns may be in different orders; the notebook validates names and reorders validation/test columns to the training order.

**Checks that run on every path:** duplicate column names are rejected before pandas can rename them; rows without a target are dropped and the target must be numeric and vary; exact duplicate rows are reported (not removed); and training is capped at 10,000 rows (Mitra's supported maximum) with the cap recorded. **Upload CSV** carves a *seeded random* holdout; there is no stratification for a continuous target.


In [ ]:
import csv
import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

NETWORK_TIMEOUT_SECONDS = 30

DATA_SOURCE = 'Sample dataset (FreshRetailNet)'  # @param ['Sample dataset (FreshRetailNet)', 'Upload CSV', 'Upload pre-split train/val/test']
TARGET_COLUMN = 'target'                         # @param {type:'string'}
DROP_COLUMNS = ''                                # @param {type:'string'}
VALIDATION_SPLIT = 0.20                          # @param {type:'number'}
SEED = 42                                        # @param {type:'integer'}

SAMPLE_REVISION = '5625a9eeca94b8c72b9ad1ec78d07ecbaa720903'
SAMPLE_ZIP_URL = f'https://raw.githubusercontent.com/kurtvalcorza/mitra-regressor-pipeline/{SAMPLE_REVISION}/examples/sample-data/freshretailnet-h7.zip'
SAMPLE_CARD_URL = f'https://github.com/kurtvalcorza/mitra-regressor-pipeline/blob/{SAMPLE_REVISION}/examples/sample-data/DATASET_CARD.md'

using_sample = DATA_SOURCE == 'Sample dataset (FreshRetailNet)'
using_presplit_data = DATA_SOURCE in {'Sample dataset (FreshRetailNet)', 'Upload pre-split train/val/test'}
test_data = None
TRAIN_ROW_CAP_APPLIED = False

def read_csv_payload(payload, label):
    text = payload.decode('utf-8-sig')
    rows = csv.reader(io.StringIO(text, newline=''))
    header = next((row for row in rows if row and not (len(row) == 1 and not row[0].strip())), [])
    seen = set()
    duplicates = []
    for name in header:
        if name in seen and name not in duplicates:
            duplicates.append(name)
        seen.add(name)
    if duplicates:
        raise ValueError(f'{label} contains duplicate column names: {duplicates}')
    return pd.read_csv(io.BytesIO(payload))

def read_presplit_upload():
    from google.colab import files
    uploaded = files.upload()
    by_base = {Path(name).name.lower(): payload for name, payload in uploaded.items()}
    required = {'train.csv', 'val.csv', 'test.csv'}
    missing = sorted(required - set(by_base))
    if missing:
        raise RuntimeError(f'Upload train.csv, val.csv, and test.csv together. Missing: {missing}')
    return (
        read_csv_payload(by_base['train.csv'], 'train.csv'),
        read_csv_payload(by_base['val.csv'], 'val.csv'),
        read_csv_payload(by_base['test.csv'], 'test.csv'),
    )

if using_sample:
    with urllib.request.urlopen(SAMPLE_ZIP_URL, timeout=NETWORK_TIMEOUT_SECONDS) as r:
        payload = r.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        names = {Path(n).name: n for n in z.namelist() if not n.endswith('/')}
        required = {'train.csv', 'val.csv', 'test.csv'}
        missing = sorted(required - set(names))
        if missing:
            raise RuntimeError(f'Sample ZIP missing: {missing}')
        train_data = read_csv_payload(z.read(names['train.csv']), 'train.csv')
        holdout_data = read_csv_payload(z.read(names['val.csv']), 'val.csv')
        test_data = read_csv_payload(z.read(names['test.csv']), 'test.csv')
    TARGET_COLUMN = 'target'
    print('✓ Using FreshRetailNet regression sample with preserved train/val/test splits.')
    print('  Sample revision:', SAMPLE_REVISION)
    print('  Dataset card:', SAMPLE_CARD_URL)
elif DATA_SOURCE == 'Upload pre-split train/val/test':
    train_data, holdout_data, test_data = read_presplit_upload()
    print('✓ Using uploaded train/val/test partitions without re-splitting.')
else:
    from google.colab import files
    uploaded = files.upload()
    csvs = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith('.csv')]
    if len(csvs) != 1:
        raise RuntimeError('Upload exactly one labelled CSV.')
    data = read_csv_payload(csvs[0][1], 'uploaded CSV')
    print('⚠ Upload CSV uses a seeded random holdout and assumes rows are IID. For temporal/grouped/lagged data, use the pre-split option.')

drop_columns = [c.strip() for c in DROP_COLUMNS.split(',') if c.strip() and c.strip() != TARGET_COLUMN]

def prepare(df, name, require_variation=False, min_rows=2):
    if df.columns.duplicated().any():
        raise ValueError(f'{name}: duplicate column names are not supported.')
    if TARGET_COLUMN not in df.columns:
        raise ValueError(f'{name}: target {TARGET_COLUMN!r} not found.')
    out = df.drop(columns=[c for c in drop_columns if c in df.columns], errors='ignore').copy()
    raw_target = out[TARGET_COLUMN]
    numeric_target = pd.to_numeric(raw_target, errors='coerce')
    non_numeric = raw_target.notna() & numeric_target.isna()
    if non_numeric.any():
        examples = raw_target[non_numeric].astype(str).head(5).tolist()
        raise ValueError(f'{name}: target must be numeric; examples of invalid values: {examples}')
    out[TARGET_COLUMN] = numeric_target
    rows_before_target_drop = len(out)
    out = out.dropna(subset=[TARGET_COLUMN]).copy()
    dropped_target_rows = rows_before_target_drop - len(out)
    if dropped_target_rows:
        print(f'⚠ {name}: dropped {dropped_target_rows:,} row(s) with a missing target.')
    if not np.isfinite(out[TARGET_COLUMN].to_numpy(dtype=float)).all():
        raise ValueError(f'{name}: target contains infinite values; use finite numeric regression targets only.')
    duplicate_rows = int(out.duplicated().sum())
    if duplicate_rows:
        print(f'⚠ {name}: {duplicate_rows:,} exact duplicate labelled rows detected; inspect for leakage or accidental copies.')
    features = [c for c in out.columns if c != TARGET_COLUMN]
    errors = []
    if len(out) < min_rows:
        errors.append(f'use at least {min_rows} labelled rows')
    if not features:
        errors.append('no feature columns remain')
    if len(features) > 500:
        errors.append(f'{len(features)} features exceed the 500-feature limit')
    if require_variation and out[TARGET_COLUMN].nunique(dropna=True) < 2:
        errors.append('training target has no variation')
    if errors:
        raise ValueError(f'{name} is not ready: ' + '; '.join(errors))
    return out, features

if using_presplit_data:
    train_data, features = prepare(train_data, 'train.csv', require_variation=True, min_rows=50)
    holdout_data, val_features = prepare(holdout_data, 'val.csv')
    test_data, test_features = prepare(test_data, 'test.csv')
    train_feature_set = set(features)
    if set(val_features) != train_feature_set or set(test_features) != train_feature_set:
        raise ValueError('train/val/test feature column names do not match.')
    ordered_columns = features + [TARGET_COLUMN]
    holdout_data = holdout_data.reindex(columns=ordered_columns)
    test_data = test_data.reindex(columns=ordered_columns)
else:
    clean, features = prepare(data, 'uploaded CSV', require_variation=True, min_rows=50)
    if not 0.05 <= VALIDATION_SPLIT <= 0.40:
        raise ValueError('VALIDATION_SPLIT must be 0.05–0.40.')
    train_data, holdout_data = train_test_split(
        clean,
        test_size=VALIDATION_SPLIT,
        random_state=SEED,
        shuffle=True,
    )
    if train_data[TARGET_COLUMN].nunique(dropna=True) < 2:
        raise ValueError('Training split has no target variation. Use pre-split data, add more varied rows, or adjust VALIDATION_SPLIT.')

TRAIN_ROWS_BEFORE_CAP = len(train_data)
if len(train_data) > 10_000:
    train_data = train_data.sample(n=10_000, random_state=SEED)
    TRAIN_ROW_CAP_APPLIED = True
    if train_data[TARGET_COLUMN].nunique(dropna=True) < 2:
        raise ValueError('Capped training split has no target variation; provide a representative pre-split training set.')

FEATURE_COLUMNS = [c for c in train_data.columns if c != TARGET_COLUMN]
PROBLEM_TYPE = 'regression'

target_summary = pd.DataFrame({
    'split': ['train', 'holdout'] + (['test'] if test_data is not None else []),
    'rows': [len(train_data), len(holdout_data)] + ([len(test_data)] if test_data is not None else []),
    'target_mean': [train_data[TARGET_COLUMN].mean(), holdout_data[TARGET_COLUMN].mean()] + ([test_data[TARGET_COLUMN].mean()] if test_data is not None else []),
    'target_std': [train_data[TARGET_COLUMN].std(), holdout_data[TARGET_COLUMN].std()] + ([test_data[TARGET_COLUMN].std()] if test_data is not None else []),
    'target_min': [train_data[TARGET_COLUMN].min(), holdout_data[TARGET_COLUMN].min()] + ([test_data[TARGET_COLUMN].min()] if test_data is not None else []),
    'target_max': [train_data[TARGET_COLUMN].max(), holdout_data[TARGET_COLUMN].max()] + ([test_data[TARGET_COLUMN].max()] if test_data is not None else []),
})
display(pd.DataFrame({
    'Item': ['Training rows', 'Holdout rows', 'Independent test rows', 'Features', 'Target', 'Problem type'],
    'Value': [len(train_data), len(holdout_data), len(test_data) if test_data is not None else None, len(FEATURE_COLUMNS), TARGET_COLUMN, PROBLEM_TYPE],
}))
display(target_summary)

zero_fraction = float((train_data[TARGET_COLUMN] == 0).mean())
if zero_fraction >= 0.50:
    print(f'⚠ Training target is {zero_fraction:.1%} zeros. Compare Mitra against a strong naive baseline; highly intermittent targets can be difficult.')
if TRAIN_ROW_CAP_APPLIED:
    print(f'⚠ Training rows capped from {TRAIN_ROWS_BEFORE_CAP:,} to {len(train_data):,}.')
if len(train_data) > 5_000:
    print("⚠ Above Mitra's particularly strong reported ≤5,000-sample regime.")
if len(FEATURE_COLUMNS) > 100:
    print("⚠ Above Mitra's particularly strong reported ≤100-feature regime.")


## 4. Evaluate pretrained Mitra, then optionally fine-tune

**What "pretrained evaluation" means here.** With `fine_tune=False`, Mitra does not update a single weight. AutoGluon hands it your training rows as *context*, and for each validation row the model predicts a value by attending over that context. `predictor.fit(...)` still runs, because that is AutoGluon's API, but what it does is register the context and pick the model configuration; the 300 MB of weights are the same bytes you verified in Step 2.

When a pre-split source is used, `val.csv` is the **holdout** (used for reporting and, if fine-tuning runs, for choosing between pretrained and fine-tuned) and `test.csv` is the **independent test**, reported but never used for any decision. That separation is what keeps the test numbers honest.

**Fine-tuning** (`RUN_FINE_TUNING`, GPU only) trains the weights for `FINE_TUNE_STEPS` steps on your training rows. The default path leaves it off, so by default this step shows a *before* without an *after*; switch it on to get the comparison table.

**Colab memory note:** `MAX_MEMORY_USAGE_RATIO=1.10` completed an end-to-end regressor run on a standard Tesla T4 Colab, but memory was tight: that run's fine-tune reduced `max_samples_support` from 8192 to 4096 to 2048 before finishing under the time limit. Keep 1.10 as a cautious setting and do **not** raise it merely to silence a memory warning: values above 1.0 intentionally accept more out-of-memory risk. If fitting is skipped or fails, first reduce rows or context, or use a higher-memory runtime.

**Fine-tuning note:** `FINE_TUNE_STEPS=50` makes the requested schedule explicit. `FINE_TUNE_TIME_LIMIT` can truncate that schedule, so a time-limited run should not be interpreted as a controlled 50-step experiment.

**Metric direction:** AutoGluon stores error metrics in higher-is-better form, so `evaluate()` returns them negated. The helper converts MAE, RMSE, MSE and their relatives back to conventional positive values; R² and the correlations stay higher-is-better, and every table says which way is better.

**Rerun safety:** this step clears predictor objects and output directories from any earlier execution before fitting, so a failed rerun cannot leave an older model eligible for inference or export. If AutoGluon still reports insufficient RAM in a fresh run, restart the session or use a higher-memory runtime rather than raising the safety ratio.

**Selection guard:** when fine-tuning ran, the notebook recommends pretrained or fine-tuned by `EVAL_METRIC` on the holdout only, and only if the holdout has at least `MIN_SELECTION_HOLDOUT_ROWS = 50` rows; below that it keeps the pretrained predictor and reports the fine-tuned metrics as evidence. A worse independent-test error after fine-tuning is printed as a warning and never changes the selection.


In [ ]:
import gc
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from autogluon.tabular import TabularPredictor

EVAL_METRIC = 'mean_absolute_error'  # @param ['mean_absolute_error', 'root_mean_squared_error']
BASELINE_TIME_LIMIT = 300            # @param {type:'integer'}
RUN_FINE_TUNING = False              # @param {type:'boolean'}
FINE_TUNE_STEPS = 50                 # @param {type:'integer'}
FINE_TUNE_TIME_LIMIT = 600           # @param {type:'integer'}
MAX_MEMORY_USAGE_RATIO = 1.10        # @param {type:'number'}
MIN_SELECTION_HOLDOUT_ROWS = 50

BASELINE_PATH = Path('/content/mitra-baseline')
FINETUNED_PATH = Path('/content/mitra-finetuned')
EXPORT_ZIP_PATH = Path('/content/mitra-predictor.zip')

FIT_RUN_COMPLETED = False
for stale_name in (
    'active_predictor',
    'recommended_predictor',
    'active_mode',
    'selection_basis',
    'baseline_predictor',
    'finetuned_predictor',
    'baseline_metrics',
    'baseline_test_metrics',
    'finetuned_metrics',
    'finetuned_test_metrics',
):
    globals().pop(stale_name, None)

for stale_path in (BASELINE_PATH, FINETUNED_PATH):
    shutil.rmtree(stale_path, ignore_errors=True)
EXPORT_ZIP_PATH.unlink(missing_ok=True)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

baseline_predictor = None
finetuned_predictor = None
baseline_metrics = None
baseline_test_metrics = None
finetuned_metrics = None
finetuned_test_metrics = None

CUDA_AVAILABLE = torch.cuda.is_available()
print('PyTorch:', torch.__version__)
print('PyTorch CUDA build:', torch.version.cuda)
print('CUDA available:', CUDA_AVAILABLE, torch.cuda.get_device_name(0) if CUDA_AVAILABLE else '')
print(f'AutoGluon memory safety ratio: {MAX_MEMORY_USAGE_RATIO:.2f}')
print('✓ Cleared stale predictor state and output paths before fitting.')

REG_LOWER_IS_BETTER = {
    'root_mean_squared_error',
    'mean_squared_error',
    'mean_absolute_error',
    'median_absolute_error',
    'mean_absolute_percentage_error',
    'symmetric_mean_absolute_percentage_error',
    'root_mean_squared_logarithmic_error',
}
MITRA_METRIC_MAP = {
    'mean_absolute_error': 'mae',
    'root_mean_squared_error': 'rmse',
}
if EVAL_METRIC not in MITRA_METRIC_MAP:
    raise ValueError(
        f'Unsupported EVAL_METRIC {EVAL_METRIC!r}. Choose one of: {sorted(MITRA_METRIC_MAP)}'
    )

def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

def fit_mitra(fine_tune, path, time_limit, steps=None):
    seed_everything()
    hp = {'fine_tune': fine_tune, 'seed': SEED}
    native_metric = MITRA_METRIC_MAP.get(EVAL_METRIC)
    if native_metric is not None:
        hp['metric'] = native_metric
    if fine_tune:
        if steps is None or steps <= 0:
            raise ValueError('FINE_TUNE_STEPS must be a positive integer when fine-tuning is enabled.')
        hp['fine_tune_steps'] = steps
    predictor = TabularPredictor(
        label=TARGET_COLUMN,
        problem_type='regression',
        eval_metric=EVAL_METRIC,
        path=str(path),
        verbosity=2,
    )
    predictor.fit(
        train_data,
        hyperparameters={'MITRA': hp},
        fit_weighted_ensemble=False,
        time_limit=time_limit,
        ag_args_fit={'max_memory_usage_ratio': MAX_MEMORY_USAGE_RATIO},
    )
    if not any('mitra' in n.lower() for n in predictor.model_names()):
        raise RuntimeError(f'Expected Mitra; AutoGluon trained {predictor.model_names()}.')
    return predictor

def metrics(predictor, frame):
    raw = predictor.evaluate(frame, auxiliary_metrics=True, silent=True)
    return {k: float(-v if k in REG_LOWER_IS_BETTER else v) for k, v in raw.items()}

def metric_table(values, name):
    frame = pd.DataFrame({'value': pd.Series(values)})
    frame['direction'] = ['lower is better' if metric in REG_LOWER_IS_BETTER else 'higher is better' for metric in frame.index]
    frame.columns = [name, 'direction']
    return frame

baseline_predictor = fit_mitra(False, BASELINE_PATH, BASELINE_TIME_LIMIT)
baseline_metrics = metrics(baseline_predictor, holdout_data)
baseline_test_metrics = metrics(baseline_predictor, test_data) if test_data is not None else None
display(metric_table(baseline_metrics, 'Pretrained — holdout'))
if baseline_test_metrics is not None:
    display(metric_table(baseline_test_metrics, 'Pretrained — independent test'))

if RUN_FINE_TUNING:
    if not CUDA_AVAILABLE:
        raise RuntimeError('Fine-tuning requires a GPU. Choose Runtime → Change runtime type → GPU.')
    finetuned_predictor = fit_mitra(
        True,
        FINETUNED_PATH,
        FINE_TUNE_TIME_LIMIT,
        FINE_TUNE_STEPS,
    )
    finetuned_metrics = metrics(finetuned_predictor, holdout_data)
    comparison = pd.DataFrame({'Pretrained': baseline_metrics, 'Fine-tuned': finetuned_metrics})
    comparison['direction'] = ['lower is better' if metric in REG_LOWER_IS_BETTER else 'higher is better' for metric in comparison.index]
    display(comparison)
    print('Interpret small metric deltas cautiously; repeat across seeds/splits when the decision matters. The time limit may truncate the requested fine-tune schedule.')
    if test_data is not None:
        finetuned_test_metrics = metrics(finetuned_predictor, test_data)
        display(metric_table(finetuned_test_metrics, 'Fine-tuned — independent test'))
else:
    print('Fine-tuning skipped. Set RUN_FINE_TUNING=True on a GPU to run it.')

def metric_is_better(candidate, baseline, metric_name):
    if metric_name not in candidate or metric_name not in baseline:
        raise RuntimeError(f'Metric {metric_name!r} was not returned by AutoGluon; cannot select a predictor safely.')
    if metric_name in REG_LOWER_IS_BETTER:
        return candidate[metric_name] < baseline[metric_name]
    return candidate[metric_name] > baseline[metric_name]

def metric_is_worse(candidate, baseline, metric_name):
    if metric_name not in candidate or metric_name not in baseline:
        return False
    if metric_name in REG_LOWER_IS_BETTER:
        return candidate[metric_name] > baseline[metric_name]
    return candidate[metric_name] < baseline[metric_name]

recommended_predictor = baseline_predictor
active_mode = 'pretrained'
selection_basis = 'default:pretrained'
if finetuned_predictor is not None:
    if len(holdout_data) < MIN_SELECTION_HOLDOUT_ROWS:
        selection_basis = (
            f'default:pretrained; holdout-too-small:'
            f'{len(holdout_data)}<{MIN_SELECTION_HOLDOUT_ROWS}'
        )
        print(
            '⚠ Holdout is too small for automatic model selection '
            f'({len(holdout_data)} rows; minimum {MIN_SELECTION_HOLDOUT_ROWS}). '
            'Keeping the pretrained predictor for inference/export. '
            'Fine-tuned metrics are still reported as evaluation evidence.'
        )
    else:
        selection_basis = f'holdout:{EVAL_METRIC}'
        if metric_is_better(finetuned_metrics, baseline_metrics, EVAL_METRIC):
            recommended_predictor = finetuned_predictor
            active_mode = 'fine-tuned'
        print(
            f'✓ Recommended predictor for inference/export: {active_mode} '
            f'(selected by {EVAL_METRIC} on the holdout: '
            f'pretrained={baseline_metrics[EVAL_METRIC]:.6g}, fine-tuned={finetuned_metrics[EVAL_METRIC]:.6g}).'
        )
    if finetuned_test_metrics is not None and baseline_test_metrics is not None:
        degraded = [
            metric_name for metric_name in baseline_test_metrics
            if metric_name in finetuned_test_metrics
            and metric_is_worse(finetuned_test_metrics, baseline_test_metrics, metric_name)
        ]
        if degraded:
            details = ', '.join(
                f'{name}: {baseline_test_metrics[name]:.6g} → {finetuned_test_metrics[name]:.6g}'
                for name in degraded
            )
            print(
                '⚠ Fine-tuning produced mixed independent-test evidence. '
                f'These metrics worsened: {details}. Selection never uses the independent test; '
                'it remains evaluation evidence only.'
            )
else:
    print('✓ Recommended predictor for inference/export: pretrained (fine-tuning not run).')

active_predictor = recommended_predictor

FIT_RUN_COMPLETED = True
print('✓ Step 4 completed successfully; this run is eligible for inference/export.')


## 4b. Companion classical tree baselines & in-memory post-hoc ensembling

Evaluate lightweight classical tree baselines (**LightGBM** and **Random Forest**) on the exact same (capped) training rows (`train_data`) and holdout partition (`holdout_data`) to benchmark Mitra's foundation model performance and demonstrate in-memory point-prediction ensembling.

In [ ]:
import time
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OrdinalEncoder

if not globals().get('FIT_RUN_COMPLETED', False) or baseline_predictor is None:
    raise RuntimeError('Run Step 4 successfully before running baseline comparisons.')

print('--- Step 4b: Classical Tree Baselines & In-Memory Ensembling (Holdout) ---')

# 1. Fit classical tree baselines with train-fitted categorical preprocessing for BYOD robustness
cat_cols = [c for c in FEATURE_COLUMNS if not pd.api.types.is_numeric_dtype(train_data[c])]
num_cols = [c for c in FEATURE_COLUMNS if pd.api.types.is_numeric_dtype(train_data[c])]

# keep_empty_features=True ensures all-NaN numeric columns in BYOD CSVs are imputed to 0.0 rather than dropped
num_imputer = SimpleImputer(strategy='median', keep_empty_features=True) if num_cols else None
cat_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1) if cat_cols else None


def fit_transform_trees(train_df):
    parts = []
    if num_cols:
        arr_num = num_imputer.fit_transform(train_df[num_cols])
        parts.append(pd.DataFrame(arr_num, columns=num_cols, index=train_df.index))
    if cat_cols:
        # Note: .astype(str) maps missing categoricals to 'nan' as an observed category
        cat_str = train_df[cat_cols].astype(str)
        arr_cat = cat_encoder.fit_transform(cat_str)
        parts.append(pd.DataFrame(arr_cat, columns=cat_cols, index=train_df.index))
    return pd.concat(parts, axis=1)[FEATURE_COLUMNS]


def transform_trees(eval_df):
    parts = []
    if num_cols:
        arr_num = num_imputer.transform(eval_df[num_cols])
        parts.append(pd.DataFrame(arr_num, columns=num_cols, index=eval_df.index))
    if cat_cols:
        cat_str = eval_df[cat_cols].astype(str)
        arr_cat = cat_encoder.transform(cat_str)
        parts.append(pd.DataFrame(arr_cat, columns=cat_cols, index=eval_df.index))
    return pd.concat(parts, axis=1)[FEATURE_COLUMNS]


X_tr_tree = fit_transform_trees(train_data)
y_tr = train_data[TARGET_COLUMN]

t0_lgbm = time.perf_counter()
lgbm_model = LGBMRegressor(
    random_state=SEED,
    n_estimators=100,
    verbose=-1,
)
lgbm_model.fit(X_tr_tree, y_tr)
t_fit_lgbm = time.perf_counter() - t0_lgbm

t0_rf = time.perf_counter()
rf_model = RandomForestRegressor(
    random_state=SEED,
    n_estimators=100,
)
rf_model.fit(X_tr_tree, y_tr)
t_fit_rf = time.perf_counter() - t0_rf


def timed_predict(model_type, model, frame, dev_name, repeats=5):
    if model_type == 'mitra':
        X = frame[FEATURE_COLUMNS]
        # Discarded warm-up call to eliminate first-call setup/autotuning
        _ = model.predict(X)
        latencies = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            pred_series = model.predict(X)
            latencies.append((time.perf_counter() - t0) * 1000.0)
        preds = np.asarray(pred_series.values, dtype=float)
    else:
        X_tree = transform_trees(frame)
        # Discarded warm-up call
        _ = model.predict(X_tree)
        latencies = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            preds = np.asarray(model.predict(X_tree), dtype=float)
            latencies.append((time.perf_counter() - t0) * 1000.0)
    lat_ms = float(np.median(latencies))
    return preds, lat_ms, dev_name


dev_mitra = 'cuda' if torch.cuda.is_available() else 'cpu'
pred_mitra_h, lat_mitra_h, _ = timed_predict('mitra', active_predictor, holdout_data, dev_mitra)
pred_lgbm_h, lat_lgbm_h, _ = timed_predict('tree', lgbm_model, holdout_data, 'cpu')
pred_rf_h, lat_rf_h, _ = timed_predict('tree', rf_model, holdout_data, 'cpu')

y_h = holdout_data[TARGET_COLUMN].to_numpy(dtype=float)
n_h = len(holdout_data)


def eval_regression(y_true, y_pred):
    mse_val = float(mean_squared_error(y_true, y_pred))
    return {
        'rmse': float(np.sqrt(mse_val)),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
    }


m_mitra_h = eval_regression(y_h, pred_mitra_h)
m_lgbm_h = eval_regression(y_h, pred_lgbm_h)
m_rf_h = eval_regression(y_h, pred_rf_h)

print(f'\nHoldout sample count: {n_h:,} rows (training cap: 10,000)')
print(f"{'Model':<20} | {'Device':<6} | {'Rows':<6} | {'RMSE':<10} | {'MAE':<10} | {'R2':<10} | {'Latency':<10}")
print('-' * 85)
print(f"{'Mitra (' + active_mode + ')':<20} | {dev_mitra:<6} | {n_h:<6} | {m_mitra_h['rmse']:<10.4f} | {m_mitra_h['mae']:<10.4f} | {m_mitra_h['r2']:<10.4f} | {lat_mitra_h:<8.2f} ms")
print(f"{'LightGBM':<20} | {'cpu':<6} | {n_h:<6} | {m_lgbm_h['rmse']:<10.4f} | {m_lgbm_h['mae']:<10.4f} | {m_lgbm_h['r2']:<10.4f} | {lat_lgbm_h:<8.2f} ms")
print(f"{'Random Forest':<20} | {'cpu':<6} | {n_h:<6} | {m_rf_h['rmse']:<10.4f} | {m_rf_h['mae']:<10.4f} | {m_rf_h['r2']:<10.4f} | {lat_rf_h:<8.2f} ms")

# 2. In-memory post-hoc convex point-prediction blend (minimizing holdout RMSE)
best_w = 1.0
best_rmse = float('inf')
grid = np.linspace(0.0, 1.0, 101)
for w in grid:
    blend_pred = w * pred_mitra_h + (1.0 - w) * pred_lgbm_h
    current_rmse = float(np.sqrt(mean_squared_error(y_h, blend_pred)))
    if current_rmse < best_rmse:
        best_rmse = current_rmse
        best_w = float(w)

pred_blend_h = best_w * pred_mitra_h + (1.0 - best_w) * pred_lgbm_h
m_blend_h = eval_regression(y_h, pred_blend_h)
rmse_delta_h = m_blend_h['rmse'] - m_mitra_h['rmse']

se_mitra_h = (y_h - pred_mitra_h) ** 2
se_blend_h = (y_h - pred_blend_h) ** 2
paired_se_h = float(np.std(se_blend_h - se_mitra_h, ddof=1) / np.sqrt(n_h)) if n_h > 1 else 0.0

print(f'\n[Post-Hoc Ensembling] Declared objective: minimize holdout RMSE')
print(f'Optimal holdout blend weight: {best_w:.2f} Mitra + {1.0 - best_w:.2f} LightGBM')
print(f"Holdout Blend: RMSE={m_blend_h['rmse']:.4f} (delta vs Mitra: {rmse_delta_h:+.4f}, paired SE of diff={paired_se_h:.4f}), MAE={m_blend_h['mae']:.4f}, R2={m_blend_h['r2']:.4f}")

# 3. Evaluate generalization on independent test set if present
if test_data is not None:
    pred_mitra_t, _, _ = timed_predict('mitra', active_predictor, test_data, dev_mitra)
    pred_lgbm_t, _, _ = timed_predict('tree', lgbm_model, test_data, 'cpu')
    pred_rf_t, _, _ = timed_predict('tree', rf_model, test_data, 'cpu')
    pred_blend_t = best_w * pred_mitra_t + (1.0 - best_w) * pred_lgbm_t
    y_t = test_data[TARGET_COLUMN].to_numpy(dtype=float)
    n_t = len(test_data)

    m_mitra_t = eval_regression(y_t, pred_mitra_t)
    m_lgbm_t = eval_regression(y_t, pred_lgbm_t)
    m_rf_t = eval_regression(y_t, pred_rf_t)
    m_blend_t = eval_regression(y_t, pred_blend_t)
    rmse_delta_t = m_blend_t['rmse'] - m_mitra_t['rmse']

    se_mitra_t = (y_t - pred_mitra_t) ** 2
    se_blend_t = (y_t - pred_blend_t) ** 2
    paired_se_t = float(np.std(se_blend_t - se_mitra_t, ddof=1) / np.sqrt(n_t)) if n_t > 1 else 0.0

    print(f'\n[Independent Test Generalization] {n_t:,} rows:')
    print(f"  Mitra Test: RMSE={m_mitra_t['rmse']:.4f}, MAE={m_mitra_t['mae']:.4f}, R2={m_mitra_t['r2']:.4f}")
    print(f"  LightGBM Test: RMSE={m_lgbm_t['rmse']:.4f}, MAE={m_lgbm_t['mae']:.4f}, R2={m_lgbm_t['r2']:.4f}")
    print(f"  Random Forest Test: RMSE={m_rf_t['rmse']:.4f}, MAE={m_rf_t['mae']:.4f}, R2={m_rf_t['r2']:.4f}")
    print(f"  Ensemble Test: RMSE={m_blend_t['rmse']:.4f} (delta vs Mitra: {rmse_delta_t:+.4f}, paired SE of diff={paired_se_t:.4f}), MAE={m_blend_t['mae']:.4f}, R2={m_blend_t['r2']:.4f}")

    if rmse_delta_h < 0 and rmse_delta_t > 0:
        print('⚠ Mixed-evidence warning: Holdout blend improved RMSE, but independent test RMSE degraded.')
        print('  Holdout weight selection may be slightly overfitted; treat holdout gain as selection-biased.')
else:
    print('ℹ No independent test partition present; holdout ensemble score is selection-biased demonstration evidence.')

# 4. In-memory fast CPU latency comparison
print(f'\n[In-Memory Latency Summary] Batch of {n_h:,} rows (median of 5 warmed runs):')
print(f'  Mitra ({dev_mitra}): {lat_mitra_h:.2f} ms')
print(f'  LightGBM (cpu): {lat_lgbm_h:.2f} ms ({lat_mitra_h / max(lat_lgbm_h, 0.01):.1f}x speedup on CPU)')
print(f'  Random Forest (cpu): {lat_rf_h:.2f} ms ({lat_mitra_h / max(lat_rf_h, 0.01):.1f}x speedup on CPU)')
print('Note: the blend is evaluated in memory only; Step 6 exports the unchanged model bundle.')

**What the baseline and ensemble numbers show.**
- **Fair comparison:** LightGBM and Random Forest train on the exact same `X_tr, y_tr` data (honoring the 10,000-row training cap) and score on the exact same holdout and test partitions.
- **Single-objective blending:** The convex blend optimizes strictly for holdout RMSE. Reporting secondary metrics (MAE, R²) alongside the one-row resolution ($1/N$) ensures small metric differences are interpreted transparently.
- **Export contract unchanged:** The post-hoc blend is evaluated strictly in-memory. The exported artifact in Step 6 remains the pure, validated AutoGluon model bundle (`mitra-predictor.zip`).

**What the numbers mean.** On the bundled sample the pretrained model reaches a **mean absolute error of about 0.41 on the holdout and 0.44 on the independent test**, with RMSE near 0.84 and R² near 0.78 on the test (development runs; yours will be close but not identical). The trivial baselines are: always predicting the training mean gives MAE 0.70 / 0.75, always predicting the median gives 0.60 / 0.71. Mitra is clearly doing real work with no training at all, and the holdout-to-test gap is the kind of drift a chronological split is supposed to reveal.

- **MAE** is in the target's units and robust to the long tail; **RMSE** punishes the large misses a skewed target produces, which is why it sits well above MAE here. Choose the one that matches what a miss costs you.
- **R²** can look modest on a skewed, noisy demand series even when MAE is clearly better than the baseline; do not read it alone.
- With fine-tuning on, the comparison table shows both runs side by side. In the development run, 50 fine-tuning steps cut the holdout MAE from 0.412 to 0.367 while the independent test's RMSE rose from 0.84 to 0.91 and its R² fell from 0.78 to 0.74, so the notebook recommended the fine-tuned predictor on the holdout and printed the mixed-evidence warning. A holdout win that the independent test contradicts is exactly the pattern to distrust: fine-tuning may have fitted the holdout period rather than the problem. Small deltas should be repeated across seeds or splits before you act on them; the notebook prints that reminder.
- These are **evaluation** numbers for this sample. Whether an MAE of 0.41 units is *good* depends on what over- and under-forecasting cost you; that judgement is yours, not the notebook's.

**Try it:** switch `EVAL_METRIC` to `root_mean_squared_error` and re-run this step; the recommended predictor may change, because the two errors weight big misses differently.


## 5. Predict new rows

Upload an unlabelled CSV with the same feature columns (order does not matter; extra columns are kept in the output but not used). For a pre-split dataset, `test.csv` was already scored above; this step is for genuinely new rows. It is off by default (`RUN_NEW_DATA_INFERENCE`) so that a top-to-bottom run needs no upload dialog.

The output adds one scalar `prediction` column; a CSV that already has a `prediction` column is rejected rather than overwritten. The cell refuses to run unless Step 4 completed in this session, which is why the flag `FIT_RUN_COMPLETED` exists.


In [ ]:
import csv
import io

import pandas as pd

def read_inference_csv(payload):
    text = payload.decode('utf-8-sig')
    rows = csv.reader(io.StringIO(text, newline=''))
    header = next((row for row in rows if row and not (len(row) == 1 and not row[0].strip())), [])
    seen = set()
    duplicates = []
    for name in header:
        if name in seen and name not in duplicates:
            duplicates.append(name)
        seen.add(name)
    if duplicates:
        raise ValueError(f'Inference CSV contains duplicate column names: {duplicates}')
    return pd.read_csv(io.BytesIO(payload))

RUN_NEW_DATA_INFERENCE = False  # @param {type:'boolean'}

if RUN_NEW_DATA_INFERENCE:
    if not globals().get('FIT_RUN_COMPLETED', False) or baseline_predictor is None:
        raise RuntimeError('No predictor was successfully trained in this Step 4 execution. Run Step 4 successfully before inference.')
    from google.colab import files
    uploaded = files.upload()
    csvs = [(name, payload) for name, payload in uploaded.items() if name.lower().endswith('.csv')]
    if len(csvs) != 1:
        raise RuntimeError('Upload exactly one inference CSV.')
    new_data = read_inference_csv(csvs[0][1])
    if new_data.columns.duplicated().any():
        duplicates = list(new_data.columns[new_data.columns.duplicated()])
        raise ValueError(f'Inference CSV contains duplicate column names: {duplicates}')
    missing = [c for c in FEATURE_COLUMNS if c not in new_data.columns]
    if missing:
        raise ValueError(f'Inference CSV is missing required features: {missing}')
    if 'prediction' in new_data.columns:
        raise ValueError("Inference CSV already contains a 'prediction' column; rename or remove it before running inference.")
    X = new_data.reindex(columns=FEATURE_COLUMNS).copy()
    active = active_predictor
    pred = active.predict(X)
    out = new_data.copy()
    out['prediction'] = pred.to_numpy()
    out.to_csv('/content/predictions.csv', index=False)
    display(out.head())
    files.download('/content/predictions.csv')
else:
    print('Inference skipped.')


## 6. Export the reusable predictor

The AutoGluon `TabularPredictor` directory is the reusable trained artifact: it holds the Mitra weights *and* the training context the model needs at prediction time, plus AutoGluon's preprocessing state. The exported ZIP is therefore not a replacement `model.safetensors`; extract it and load the directory with `TabularPredictor.load(path)`. Reload it with the same runtime recorded in `tutorial_run_metadata.json` (this tutorial pins `autogluon.tabular[mitra]==1.5.0`).

`tutorial_run_metadata.json` is written into the predictor directory so the bundle explains itself: checkpoint identity and digests, runtime versions, feature list, data source and row counts, the metric used, whether fine-tuning ran, the selection basis, and every metric table from Step 4.

The export cell refuses to package a predictor unless the **current Step 4 execution** completed successfully, so an older in-memory predictor or a leftover directory cannot be mistaken for the result of a failed rerun.

**Trust boundary, for whoever receives the ZIP:** `TabularPredictor.load` deserializes Python objects (pickle). The companion inference notebook checks the archive's SHA-256 when you give it one, and refuses unsafe archive paths, but nothing makes an untrusted predictor archive safe to load. Print and keep the ZIP digest.


In [ ]:
import importlib.metadata as importlib_metadata
import json
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

if not globals().get('FIT_RUN_COMPLETED', False) or baseline_predictor is None:
    raise RuntimeError(
        'No predictor was successfully trained in this Step 4 execution. '
        'Run Step 4 successfully before exporting.'
    )

active_path = Path(active_predictor.path)
if not active_path.exists() or not any('mitra' in name.lower() for name in active_predictor.model_names()):
    raise RuntimeError('The current predictor artifact is missing or does not contain Mitra; refusing to export.')

metadata = {
    'base_model': MODEL_ID,
    'base_model_revision': PINNED_REVISION,
    'weights_sha256': EXPECTED_WEIGHTS_SHA256,
    'config_sha256': EXPECTED_CONFIG_SHA256,
    'model_source': MODEL_SOURCE,
    'autogluon_version': importlib_metadata.version('autogluon.tabular'),
    'torch_version': torch.__version__,
    'torch_cuda_version': torch.version.cuda,
    'python_version': sys.version.split()[0],
    'cuda_available': torch.cuda.is_available(),
    'exported_at_utc': datetime.now(timezone.utc).isoformat(),
    'mode': active_mode,
    'selection_basis': selection_basis,
    'problem_type': 'regression',
    'target_column': TARGET_COLUMN,
    'features': FEATURE_COLUMNS,
    'seed': SEED,
    'data_source': DATA_SOURCE,
    'sample_revision': SAMPLE_REVISION if using_sample else None,
    'sample_dataset_card': SAMPLE_CARD_URL if using_sample else None,
    'train_rows_before_cap': TRAIN_ROWS_BEFORE_CAP,
    'train_rows_used': len(train_data),
    'train_row_cap_applied': TRAIN_ROW_CAP_APPLIED,
    'holdout_rows': len(holdout_data),
    'independent_test_rows': len(test_data) if test_data is not None else None,
    'eval_metric': EVAL_METRIC,
    'baseline_time_limit_seconds': BASELINE_TIME_LIMIT,
    'fine_tuning_requested': RUN_FINE_TUNING,
    'fine_tune_steps_requested': FINE_TUNE_STEPS if RUN_FINE_TUNING else None,
    'fine_tune_time_limit_seconds': FINE_TUNE_TIME_LIMIT if RUN_FINE_TUNING else None,
    'fine_tune_schedule_may_be_truncated_by_time_limit': bool(RUN_FINE_TUNING),
    'max_memory_usage_ratio': MAX_MEMORY_USAGE_RATIO,
    'holdout_metrics_pretrained': baseline_metrics,
    'holdout_metrics_finetuned': finetuned_metrics,
    'independent_test_metrics_pretrained': baseline_test_metrics,
    'independent_test_metrics_finetuned': finetuned_test_metrics,
    'ai_assistance': {
        'model_configuration': 'GPT-5.6 Sol High',
        'provider_client': 'OpenAI / ChatGPT',
        'agent_relay_role': 'Builder',
        'note': 'Attribution is provenance, not sign-off or independent verification.',
    },
}
(active_path / 'tutorial_run_metadata.json').write_text(json.dumps(metadata, indent=2))
Path('/content/mitra-predictor.zip').unlink(missing_ok=True)
archive = shutil.make_archive('/content/mitra-predictor', 'zip', root_dir=active_path)
archive_digest = sha256_file(archive)
print('✓ Predictor archive:', archive)
print('SHA-256 (save this if you will verify the ZIP later):', archive_digest)


## 7. Reload smoke test

Before treating the ZIP as reusable, this cell extracts the **exported archive** into a fresh directory, reloads it with `TabularPredictor.load(...)`, and confirms that numeric predictions match the in-memory predictor on a small holdout sample, compared with `np.allclose` at a relative tolerance of 1e-6, which is the level at which floating-point summation order, not the model, explains a difference.

This checks the actual packaging boundary users will rely on later, which is the check most tutorials skip.


In [ ]:
import shutil
import stat
import zipfile
from pathlib import Path

RELOAD_DIR = Path('/content/mitra-predictor-reload')
if RELOAD_DIR.exists():
    shutil.rmtree(RELOAD_DIR)
RELOAD_DIR.mkdir(parents=True)

with zipfile.ZipFile(archive) as z:
    reload_root = RELOAD_DIR.resolve()
    for info in z.infolist():
        member = Path(info.filename)
        if member.is_absolute() or '..' in member.parts:
            raise RuntimeError(f'Unsafe archive member path during reload smoke test: {info.filename!r}')
        mode = (info.external_attr >> 16) & 0o170000
        if mode == stat.S_IFLNK:
            raise RuntimeError(f'Symlink entries are not allowed during reload smoke test: {info.filename!r}')
        target = (reload_root / member).resolve()
        if target != reload_root and reload_root not in target.parents:
            raise RuntimeError(f'Archive member escapes reload root: {info.filename!r}')
    z.extractall(RELOAD_DIR)

reloaded_predictor = TabularPredictor.load(str(RELOAD_DIR))
smoke_X = holdout_data[FEATURE_COLUMNS].head(5).copy()

expected_pred = active_predictor.predict(smoke_X).reset_index(drop=True)
reloaded_pred = reloaded_predictor.predict(smoke_X).reset_index(drop=True)
if not np.allclose(expected_pred.to_numpy(dtype=float), reloaded_pred.to_numpy(dtype=float), rtol=1e-6, atol=1e-8):
    raise RuntimeError('Reload smoke test failed: regression predictions changed after ZIP export/reload.')

print('✓ Exported predictor ZIP reloads successfully and reproduces smoke-test predictions.')


## What a successful run proves, and what to do next

If every cell ran, this session has shown: the checkpoint you evaluated is the pinned release, byte for byte; your data passed the leakage and coverage checks; Mitra's in-context accuracy on your holdout (and, if you switched it on, after fine-tuning); and a predictor ZIP that reloads and reproduces its predictions. It has **not** shown that the model is fit for your decision. That needs your own held-out data from the period or population you will deploy on, a cost-aware metric, and, for consequential uses, subgroup and drift checks.

### Recap against the objectives
- *Acquire and verify:* Step 2 (revision + SHA-256 + resolver lock).
- *Prepare data for honest evaluation:* Step 3 (preserved splits, duplicate and class-coverage checks, row cap).
- *Evaluate in context and read the numbers:* Step 4 and the notes after it.
- *Export a self-describing bundle:* Steps 6–7.

### Next experiments, in the order they teach the most
1. Set `RUN_FINE_TUNING = True` on a GPU runtime and compare the two rows of the table; then set `FINE_TUNE_STEPS = 200` and see whether the holdout keeps improving while the independent test does not (that gap is overfitting to the holdout).
2. Change `EVAL_METRIC` to `root_mean_squared_error`; note whether the recommended predictor changes.
3. Upload your own single CSV, then the same data as pre-split files with a time-based split, and compare the two errors. The difference is the leakage the random split hides.
4. Feed the exported ZIP to the companion [inference notebook](mitra_regressor_predictor_inference_colab.ipynb) with a fresh CSV.

### Troubleshooting
| Symptom | Cause | What to do |
|---|---|---|
| `checksum mismatch` in Step 2 | the uploaded ZIP or download is not the pinned release | re-download; never edit the expected digest |
| `Offline checkpoint lock is not in effect` | `huggingface_hub` was imported before the cache was configured (cells run out of order) | *Runtime ▸ Restart session*, run from Step 1 |
| `… is not ready: …target…` | the target column is missing, non-numeric, or constant | pick the right column, or clean it |
| AutoGluon skips Mitra for memory / `Expected Mitra; AutoGluon trained […]` | the memory guard refused the fit | fewer rows or features, a higher-memory runtime; raise `MAX_MEMORY_USAGE_RATIO` only as a last resort |
| `No predictor was successfully trained in this Step 4 execution` | Step 5/6 run before or after a failed Step 4 | re-run Step 4 successfully first |
| `Predictor was exported with AutoGluon X, but this runtime has Y` (inference notebook) | version drift | install the recorded version |

### Licences and provenance
The sample is CC BY 4.0 (FreshRetailNet-50K derivative); Mitra is Apache-2.0 from the AutoGluon team at AWS; DIMER redistributes the pinned `model.safetensors` and is not the model developer. All of it is recorded in `tutorial_run_metadata.json` inside the bundle.


## AI use and provenance

This tutorial was developed with substantial AI assistance under human direction and review.

- Original build: **GPT-5.6 Sol High** (OpenAI / ChatGPT), Agent Relay role: **Builder**
- Content revision (structure, explanations, troubleshooting): **Claude Fable 5.1** (Anthropic / Claude Code), Agent Relay role: **Reviewer and Builder**
- Base-model developer: **AutoGluon team, Amazon Web Services (AWS)**
- DIMER role: distributor of the pinned `model.safetensors` artifact, not model developer

AI attribution is **provenance, not sign-off** and does not independently verify correctness. Executed checks and reproducible outputs remain the evidence for a particular run.
